# Preparing OSM training data

In this notebook, named food and retail establishments will be downloaded from OpenStreetMap within England.

Since OSM has tags and labels, we can use this to construct provisional bakery and non-bakery labels and these can be cleaned and retained for an intial machine learning input for testing data.

Records with the tag 'shop=bakery' will form our positive class and other comparable cafes, restaurants and food retailers will be our negative class. 

In [ ]:
from datetime import datetime
import json
from pathlib import Path

import pandas as pd
import requests

RAW_FOLDER = Path("../data/raw")
INTERIM_FOLDER = Path("../data/interim")

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
INTERIM_FOLDER.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = datetime.now().strftime("%Y-%m-%d")

OVERPASS_URL = ("https://overpass-api.de/api/interpreter")

In [10]:
positive_shop_values = {"bakery"}

negative_amenity_values = {"cafe", "restaurant", "fast_food"}

negative_shop_values = {"supermarket", "convenience", "butcher", "deli"}

In [11]:
OVERPASS_QUERY = """
[out:json][timeout:700];

area
    ["ISO3166-2"="GB-ENG"]
    ["admin_level"="4"]
    ->.england;

(
    nwr["shop"="bakery"](area.england);

    nwr["amenity"~"^(cafe|restaurant|fast_food)$"](area.england);

    nwr["shop"~"^(supermarket|convenience|butcher|deli)$"](area.england);
);

out tags;
"""

In [20]:
response = requests.post(OVERPASS_URL,
    data={"data": OVERPASS_QUERY},
    headers={"User-Agent": ("london-bakery-provision-dissertation")},
    timeout=300)

response.raise_for_status()

osm_payload = response.json()

osm_elements = osm_payload.get("elements",[])

if not osm_elements:
    raise RuntimeError("No OSM business records were returned.")

print(f"OSM elements downloaded: {len(osm_elements)}")

OSM elements downloaded: 148575


In [21]:
raw_path = (RAW_FOLDER/ (f"england_osm_training_raw_{SNAPSHOT_DATE}.json"))

with raw_path.open("w", encoding="utf-8") as file:
    json.dump(osm_payload, file, ensure_ascii=False, indent=2)

print(f"Raw OSM response saved to: {raw_path}")

Raw OSM response saved to: ..\data\raw\england_osm_training_raw_2026-07-27.json


In [22]:
osm_records = []

for element in osm_elements:
    tags = element.get("tags", {})

    osm_records.append({
        "Name": tags.get("name"),
        "Brand": tags.get("brand"),
        "Operator": tags.get("operator"),
        "Shop": tags.get("shop"),
        "Amenity": tags.get("amenity"),
        })

osm_businesses = pd.DataFrame(osm_records)

print("OSM business dataset shape:", osm_businesses.shape)

osm_businesses.head()

OSM business dataset shape: (148575, 5)


,Name,Brand,Operator,Shop,Amenity
0,Spar,NaN,NaN,convenience,NaN
1,Central Restaurant,NaN,NaN,NaN,restaurant
2,Morrisons Daily,Morrisons Daily,McColl's,convenience,NaN
3,The Birdham,NaN,NaN,NaN,restaurant
4,Bellini's,NaN,NaN,NaN,restaurant


# Creating a singular name to allow for FHRS comparison

In [23]:
osm_businesses["NameForModel"] = (osm_businesses["Name"].fillna(osm_businesses["Brand"]).fillna(osm_businesses["Operator"]))

bakery_records = osm_businesses[osm_businesses["Shop"].eq("bakery")].copy()

print(f"Total OSM records: {len(osm_businesses)}")

print(f"Records with a usable name: {osm_businesses['NameForModel'].notna().sum()}")

print(f"Bakery locations: {len(bakery_records)}")

print(f"Named bakery locations: {bakery_records['NameForModel'].notna().sum()}")

print(f"Unique bakery names before cleaning: {bakery_records['NameForModel'].str.casefold().nunique(dropna=True)}")

Total OSM records: 148575
Records with a usable name: 144330
Bakery locations: 4107
Named bakery locations: 4029
Unique bakery names before cleaning: 2399


# Removing records with no applicable name

In [24]:
osm_clean = osm_businesses.copy()

osm_clean["NameForModel"] = (osm_clean["Name"].fillna(osm_clean["Brand"]).fillna(osm_clean["Operator"]))

print(f"Rows before removing unnamed records: {len(osm_clean)}")

osm_clean = (osm_clean.dropna(subset=["NameForModel"]).copy())

print(f"Rows with a usable name: {len(osm_clean)}")

Rows before removing unnamed records: 148575
Rows with a usable name: 144330


# Inspecting categories to compare positive and negative classes available

In [27]:
osm_clean["BakeryLabel"] = (osm_clean["Shop"].eq("bakery").astype(int))

def assign_category_label(row):
    if row["Shop"] == "bakery":
        return "shop=bakery"

    if row["Amenity"] in negative_amenity_values:
        return f"amenity={row['Amenity']}"

    if row["Shop"] in negative_shop_values:
        return f"shop={row['Shop']}"

    return "unknown"

osm_clean["CategoryLabel"] = (osm_clean.apply(assign_category_label,axis=1,))

print(osm_clean["CategoryLabel"].value_counts())

print("\nClass counts:")

print(osm_clean["BakeryLabel"].value_counts().rename({0: "Non-bakery locations",1: "Bakery locations",}))

CategoryLabel
amenity=fast_food     37771
amenity=cafe          31553
amenity=restaurant    30921
shop=convenience      27410
shop=supermarket       8469
shop=bakery            4029
shop=butcher           2950
shop=deli              1227
Name: count, dtype: int64

Class counts:
BakeryLabel
Non-bakery locations    140301
Bakery locations          4029
Name: count, dtype: int64


# Checking for business names cleaning requirements

In [28]:
special_character_counts = (
    osm_clean["NameForModel"]
    .astype("string")
    .str.findall(r"[^\w\s]")
    .dropna()
    .explode()
    .value_counts()
)

special_character_counts

NameForModel
'    15954
&     8478
-     4284
.     1429
’      501
@      243
)      231
(      230
,      207
!      148
+      147
/      129
#      113
"       33
:       31
‘       22
°       19
;       15
?       14
%        9
`        8
|        5
£        5
•        5
~        4
–        3
[        3
]        3
=        3
$        3
*        3
—        2
（        2
№        2
¡        2
）        1
…        1
́        1
´        1
®        1
⚓        1
\        1
‎        1
Name: count, dtype: int64

# Cleaning OSM dataset to prepare for training

Including business name standardisation and removing duplicates and conflicting names

In [29]:
def clean_business_names(names):
    return (names
        .astype("string")
        .str.casefold()
        .str.replace("&", " and ", regex=False)
        .str.replace("_", " ", regex=False)
        .str.replace(r"[^\w\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

osm_clean["CleanBusinessName"] = (clean_business_names(osm_clean["NameForModel"]))

osm_clean = osm_clean[osm_clean["CleanBusinessName"].notna() & osm_clean["CleanBusinessName"].ne("")].copy()

In [32]:
labels_per_name = (osm_clean.groupby("CleanBusinessName")["BakeryLabel"].nunique())

conflicting_names = (labels_per_name[labels_per_name > 1].index)

print(f"Conflicting cleaned names: {len(conflicting_names)}")

conflicting_name_rows = (
    osm_clean.loc[osm_clean["CleanBusinessName"].isin(conflicting_names),
        ["NameForModel", "CleanBusinessName", "Shop", "Amenity", "BakeryLabel"],
    ].sort_values(["CleanBusinessName", "BakeryLabel", "NameForModel"])
)

conflicting_name_rows.head(50)

Conflicting cleaned names: 322


,NameForModel,CleanBusinessName,Shop,Amenity,BakeryLabel
137736,1066,1066,NaN,cafe,0
91960,1066,1066,bakery,NaN,1
135669,Adam's,adam s,NaN,cafe,0
33926,Adam’s,adam s,bakery,NaN,1
34313,Alberto's,alberto s,NaN,fast_food,0
126151,Alberto's,alberto s,bakery,NaN,1
80194,Allen's Bakery,allen s bakery,convenience,NaN,0
40847,Allen's Bakery,allen s bakery,bakery,NaN,1
26988,Alma,alma,NaN,cafe,0
63520,Alma,alma,NaN,cafe,0


In [33]:
osm_consistent = osm_clean[~osm_clean["CleanBusinessName"].isin(conflicting_names)].copy()

osm_training_data = (osm_consistent.groupby(
    ["CleanBusinessName","BakeryLabel"],as_index=False,)
    .agg(
        DisplayName=("NameForModel", "first"),
        LocationCount=("NameForModel", "size"),
        SourceCategories=("CategoryLabel",lambda values: " | ".join(sorted(set(values)))),
        ).sort_values(
            ["BakeryLabel","LocationCount","CleanBusinessName"],
            ascending=[False, False, True],
            ignore_index=True,
            )
            )

In [34]:
print(f"Original OSM business establishments: {len(osm_businesses)}")

print(f"Usable named business establishments: {len(osm_clean)}")

print(f"Unique establishment names to be used in model: {len(osm_training_data)}\n")

print(osm_training_data["BakeryLabel"]
      .value_counts()
      .rename({0: "Unique non-bakery business names",1: "Unique bakery business names",})
      )

Original OSM business establishments: 148575
Usable named business establishments: 144329
Unique establishment names to be used in model: 82894

BakeryLabel
Unique non-bakery business names    80831
Unique bakery business names         2063
Name: count, dtype: int64


In [35]:
TRAINING_DATA_PATH = (INTERIM_FOLDER/ (f"england_osm_training_names_{SNAPSHOT_DATE}.csv"))

osm_training_data.to_csv(TRAINING_DATA_PATH, index=False)

print(f"Model-ready training data saved to: {TRAINING_DATA_PATH}")

Model-ready training data saved to: ..\data\interim\england_osm_training_names_2026-07-27.csv
